# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sukhman-builds/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [26]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

login(token=hf_token)

print("Hugging Face connected successfully!")

Hugging Face connected successfully!


**Unit of analysis:** One row represents the daily performance of one content item for one client, identified by report_date, client_hash_id, and content_hash_id.

**Time window:** March 2026 (2026-03-01 to 2026-03-31).

**Source table:** fact_content_daily_performance.

The actual date range and uniqueness of these records will be verified using SQL queries.



In [22]:
!pip install -q duckdb


import duckdb
from huggingface_hub import hf_hub_download
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=hf_token
)

con = duckdb.connect()

schema = con.execute(
    "DESCRIBE SELECT * FROM read_parquet(?)",
    [file_path]
).df()

display(schema)


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 2. Fields: feature / label / context / excluded

### Feature
- `gsc_clicks`: Used to calculate average daily clicks.
- `gsc_impressions`: Used to calculate average daily impressions and CTR.
- `gsc_avg_position`: Used to calculate average search position.
- Derived `ctr`: Calculated from March clicks and impressions.
- Derived `observed_days`: Number of days with available GSC data.

### Label
- April `gsc_clicks`: Used only to calculate the future decline label.
- `decline_label`: Binary target indicating a 20% or greater decline.

### Context
- `report_date`: Defines the observation and outcome periods.
- `client_hash_id`, `content_hash_id`: Identify each client-content pair for grouping and joining.
- `gsc_data_available`: Filters records with available GSC data.
- `gsc_data_start`: Used to understand differences in client history.
- `ga4_data_available`: Used to inspect GA4 coverage.

### Excluded
- GA4 performance metrics: Excluded from modelling because availability is very limited.
- Client and content IDs: Not used as predictive features.
- Future April metrics: Never used as model features because they would cause data leakage.
- Private URLs and query-level information: Not needed for this assignment.

## 3. Verify it with queries (grain, counts, missing values, windows)

**Verification results (March 2026):**
- Grain check: 9,841,378 total rows and 9,841,378 unique grains. The grain is unique.
- Date range: 2026-03-01 to 2026-03-31.
- GSC availability: 3,611,061 out of 9,841,378 rows have `gsc_data_available IS TRUE`.

In [24]:
grain_check = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date) AS unique_dates,
    COUNT(DISTINCT client_hash_id) AS unique_clients,
    COUNT(DISTINCT content_hash_id) AS unique_contents
FROM read_parquet(?)
""", [file_path]).df()

print(grain_check.to_string(index=False))
grain_check = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (
        report_date,
        client_hash_id,
        content_hash_id
    )) AS unique_grains
FROM read_parquet(?)
""", [file_path]).df()

print(grain_check.to_string(index=False))

date_check = con.execute("""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(?)
""", [file_path]).df()

print(date_check.to_string(index=False))

availability_check = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS available_rows
FROM read_parquet(?)
""", [file_path]).df()

print(availability_check.to_string(index=False))

content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset",
    token=hf_token
)

content_schema = con.execute(
    "DESCRIBE SELECT * FROM read_parquet(?)",
    [content_path]
).df()

display(content_schema)
clients_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_clients.parquet",
    repo_type="dataset",
    token=hf_token
)

clients_schema = con.execute(
    "DESCRIBE SELECT * FROM read_parquet(?)",
    [clients_path]
).df()

display(clients_schema)
client_history = con.execute("""
SELECT
    COUNT(*) AS total_clients,
    COUNT(*) FILTER (
        WHERE gsc_data_start IS NOT NULL
    ) AS clients_with_gsc_start,
    MIN(gsc_data_start) AS earliest_gsc_start,
    MAX(gsc_data_start) AS latest_gsc_start
FROM read_parquet(?)
""", [clients_path]).df()

print(client_history.to_string(index=False))
march_history = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ) AS both_available_rows
FROM read_parquet(?)
""", [file_path]).df()

print(march_history.to_string(index=False))
print(file_path)
from pathlib import Path

april_path = str(Path(file_path).parents[1] / "month=2026-04" / "data_0.parquet")

print(april_path)
print(Path(april_path).exists())
from huggingface_hub import list_repo_files

april_files = [
    f for f in list_repo_files(
        "FlyRank/internship-warehouse",
        repo_type="dataset",
        token=hf_token
    )
    if "month=2026-04" in f
]

print(april_files)
from huggingface_hub import hf_hub_download

april_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=hf_token
)

print("April downloaded:", Path(april_path).exists())
print(april_path)
april_check = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows
FROM read_parquet(?)
""", [april_path]).df()

print(april_check.to_string(index=False))
overlap_check = con.execute("""
WITH march AS (
    SELECT client_hash_id, content_hash_id
    FROM read_parquet(?)
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
april AS (
    SELECT client_hash_id, content_hash_id
    FROM read_parquet(?)
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    (SELECT COUNT(*) FROM march) AS march_content_pairs,
    (SELECT COUNT(*) FROM april) AS april_content_pairs,
    COUNT(*) AS overlapping_pairs
FROM march
INNER JOIN april
USING (client_hash_id, content_hash_id)
""", [file_path, april_path]).df()

print(overlap_check.to_string(index=False))

monthly_clicks = con.execute("""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(DISTINCT report_date) AS march_days,
        SUM(gsc_clicks) * 1.0 /
            COUNT(DISTINCT report_date) AS march_avg_clicks
    FROM read_parquet(?)
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(DISTINCT report_date) AS april_days,
        SUM(gsc_clicks) * 1.0 /
            COUNT(DISTINCT report_date) AS april_avg_clicks
    FROM read_parquet(?)
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    COUNT(*) AS common_pairs,
    COUNT(*) FILTER (
        WHERE march_days >= 20
          AND april_days >= 20
          AND march_avg_clicks > 0
    ) AS eligible_pairs,
    COUNT(*) FILTER (
        WHERE march_days >= 20
          AND april_days >= 20
          AND march_avg_clicks > 0
          AND april_avg_clicks <= march_avg_clicks * 0.8
    ) AS candidate_declining_pairs
FROM march
INNER JOIN april
USING (client_hash_id, content_hash_id)
""", [file_path, april_path]).df()

print(monthly_clicks.to_string(index=False))
features = con.execute("""
SELECT
    client_hash_id,
    content_hash_id,

    AVG(gsc_clicks) AS avg_daily_clicks,
    AVG(gsc_impressions) AS avg_daily_impressions,
    AVG(gsc_avg_position) AS avg_position,
    SUM(gsc_clicks) * 1.0 /
        NULLIF(SUM(gsc_impressions), 0) AS ctr,
    COUNT(DISTINCT report_date) AS observed_days

FROM read_parquet(?)
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""", [file_path]).df()

print(features.head())
print("Rows:", len(features))


import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

labels = con.execute("""
WITH march AS (
    SELECT client_hash_id, content_hash_id,
           COUNT(DISTINCT report_date) AS march_days,
           AVG(gsc_clicks) AS march_avg
    FROM read_parquet(?)
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
april AS (
    SELECT client_hash_id, content_hash_id,
           COUNT(DISTINCT report_date) AS april_days,
           AVG(gsc_clicks) AS april_avg
    FROM read_parquet(?)
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    march.client_hash_id,
    march.content_hash_id,
    CASE
        WHEN april.april_avg <= march.march_avg * 0.8
        THEN 1 ELSE 0
    END AS decline_label
FROM march
JOIN april USING (client_hash_id, content_hash_id)
WHERE march.march_days >= 20
  AND april.april_days >= 20
  AND march.march_avg > 0
""", [file_path, april_path]).df()

dataset = features.merge(
    labels,
    on=["client_hash_id", "content_hash_id"]
)

feature_cols = [
    "avg_daily_clicks",
    "avg_daily_impressions",
    "avg_position",
    "ctr",
    "observed_days"
]

X = dataset[feature_cols].fillna(0)
y = dataset["decline_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Honest model
model = RandomForestClassifier(
    n_estimators=100, random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)
honest_score = accuracy_score(y_test, model.predict(X_test))

# Deliberately leaked model
X_leaky = X.copy()
X_leaky["future_label_leak"] = y

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaky, y, test_size=0.2, random_state=42, stratify=y
)

leaky_model = RandomForestClassifier(
    n_estimators=100, random_state=42, n_jobs=-1
)
leaky_model.fit(Xl_train, yl_train)
leaky_score = accuracy_score(yl_test, leaky_model.predict(Xl_test))

# Remove the leaked column; keep only honest features
del X_leaky

print("Dataset rows:", len(dataset))
print("Honest accuracy:", round(honest_score, 4))
print("Leaky accuracy:", round(leaky_score, 4))
print("Final features:", feature_cols)

ranking = pd.DataFrame({
    "actual": y_test.to_numpy(),
    "score": model.predict_proba(X_test)[:, 1]
}).sort_values("score", ascending=False)

print("Precision@100:", round(ranking.head(100)["actual"].mean(), 4))

 total_rows  unique_dates  unique_clients  unique_contents
    9841378            31              55           331437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  unique_grains
    9841378        9841378
 row_count first_date  last_date
   9841378 2026-03-01 2026-03-31
 total_rows  available_rows
    9841378         3611061


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None


 total_clients  clients_with_gsc_start earliest_gsc_start latest_gsc_start
           104                      67         2025-01-27       2026-06-02
 total_rows  gsc_available_rows  ga4_available_rows  both_available_rows
    9841378             3611061              413966               364347
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-04/data_0.parquet
True
['fact_content_daily_performance/month=2026-04/data_0.parquet']
April downloaded: True
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-04/data_0.parquet
 total_rows first_date  last_date  gsc_available_rows
   10424730 2026-0

... (2 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            client_hash_id           content_hash_id  avg_daily_clicks  \
0  client_62f4a7e64f5e0096  content_2e6360ad20fd7107          0.032258   
1  client_62f4a7e64f5e0096  content_ac8663da7484669a          0.000000   
2  client_62f4a7e64f5e0096  content_d49a012dcb924e31          0.000000   
3  client_62f4a7e64f5e0096  content_614baf2af4330bd7          0.032258   
4  client_62f4a7e64f5e0096  content_4a1ca0fa5c177e0c          0.000000   

   avg_daily_impressions  avg_position       ctr  observed_days  
0              29.000000      5.145765  0.001112             31  
1               2.000000      4.909314  0.000000             17  
2              10.612903      5.177774  0.000000             31  
3              24.903226      4.685335  0.001295             31  
4               1.400000      4.266667  0.000000             10  
Rows: 176738


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset rows: 59126
Honest accuracy: 0.5929
Leaky accuracy: 1.0
Final features: ['avg_daily_clicks', 'avg_daily_impressions', 'avg_position', 'ctr', 'observed_days']
Precision@100: 0.82


## 4. Data limits
### Target / Label

The target is a binary indicator of content decline.

- Observation period: March 2026.
- Outcome period: April 2026.
- A content-client pair is eligible when GSC data is available for at least 20 days in both months and March average daily clicks are greater than zero.
- Label = 1 when April average daily clicks are at least 20% lower than March average daily clicks; otherwise label = 0.
- Eligible pairs: 59,126.
Decline rate among eligible pairs: 60.4%.
- Candidate declining pairs: 35,694.

This is a future outcome proxy, not proof that a page needs refreshing.

### Features

All five features are calculated using March 2026 data, available before the April outcome.

| Feature | Why it is knowable at decision time |
|---|---|
| avg_daily_clicks | Calculated from March GSC clicks |
| avg_daily_impressions | Calculated from March GSC impressions |
| avg_position | Calculated from March search position |
| ctr | Calculated from March clicks and impressions |
| observed_days | Counts March dates with available GSC data |

### Leakage experiment

I deliberately added `decline_label`, the future outcome, as a model feature.

- Honest model accuracy: 59.83%
- Leaky model accuracy: 100%
Precision@100: 80% (the top 100 ranked pages include 80 actual declining pages).

The perfect score is artificial because the model was given the answer. The leaked feature was removed, and the final feature set contains only the five March-based features.

### Limitation

The target is a proxy based on a 20% drop in average daily clicks. It does not prove that content needs refreshing. GSC availability and historical coverage also . The target is a proxy based on a 20% drop in average daily clicks. It does not prove that content needs refreshing. GSC availability and historical coverage vary across clients, which may introduce selection bias.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.